In [28]:
# import subprocess, time, re

# # Clean up anything left over from a previous run (safe even on a fresh VM)
# subprocess.run(["pkill", "-9", "-f", "cloudflared"])
# subprocess.run(["pkill", "-9", "-f", "ollama serve"])
# time.sleep(2)

# # 1. Install deps + Ollama
# subprocess.run(["apt-get", "install", "-y", "zstd"], check=True)
# subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)

# # 2. Start Ollama server
# subprocess.Popen(["ollama", "serve"])
# time.sleep(5)

# # 3. Pull models
# for model in ["phi4-mini", "llama3.2", "qwen2.5:1.5b", "gemma4:e2b", "nomic-embed-text"]:
#     subprocess.run(["ollama", "pull", model], check=True)

# # 4. Download + start the tunnel
# subprocess.run(["rm", "-f", "cloudflared"])
# subprocess.run(
#     "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared",
#     shell=True, check=True
# )
# proc = subprocess.Popen(
#     ["./cloudflared", "tunnel", "--url", "http://localhost:11434",
#      "--http-host-header", "localhost:11434"],
#     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
# )
# time.sleep(8)
# for _ in range(20):
#     line = proc.stdout.readline()
#     print(line, end="")
#     match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
#     if match:
#         print("\nPUBLIC URL:", match.group(0))

2026-11-11T10:22:43Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-11-11T10:22:43Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-11-11T10:22:49Z INF +--------------------------------------------------------------------------------------------+
2026-11-11T10:22:49Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-11-11T10:22:49Z INF |  https://environments-degrees-stat-expiration.trycloud

In [36]:
# !nvidia-smi --query-gpu=timestamp,memory.used,memory.total,utilization.gpu --format=csv -l 2

In [38]:
import subprocess, time

# Clean up anything left over from a previous run (safe even on a fresh VM)
subprocess.run(["pkill", "-9", "-f", "ollama serve"])
time.sleep(2)

# 1. Install deps + Ollama
subprocess.run(["apt-get", "install", "-y", "zstd"], check=True)
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)

# 2. Start Ollama server
subprocess.Popen(["ollama", "serve"])
time.sleep(5)

# 3. Pull models
for model in ["phi4-mini", "llama3.2", "qwen2.5:1.5b", "gemma4:e2b", "nomic-embed-text"]:
    subprocess.run(["ollama", "pull", model], check=True)

# 4. Start the ngrok tunnel with the Host-header fix
subprocess.run(["pip", "install", "pyngrok", "-q"], check=True)
from pyngrok import ngrok
try:
    ngrok.set_auth_token("3JOrY7NmziEIwpNrGcFYkvzJ9sz_Ukexxz5guqzqbSdqLGUD")
except Exception as e:
    print("Ignoring (likely already set):", e)

policy = {
    "on_http_request": [
        {
            "actions": [
                {
                    "type": "add-headers",
                    "config": {"headers": {"host": "localhost:11434"}}
                }
            ]
        }
    ]
}

public_url = ngrok.connect(11434, "http", traffic_policy=policy)
print("PUBLIC URL:", public_url)

Ignoring (likely already set): An error occurred when saving the auth token: b'Authtoken already set in configuration file: /root/.config/ngrok/ngrok.yml\n'
PUBLIC URL: NgrokTunnel: "https://recipient-duty-untwist.ngrok-free.dev" -> "http://localhost:11434"
